In [8]:
import os
import time
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    label_binarize
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    auc
)

from sklearn.model_selection import train_test_split


# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------
DATASET_PATH = "/home/feliciano/Documents/DATASET_SEABREAM_SEGMENTED/Segmented_2s"
SAMPLE_RATE = 8000
N_MELS = 128


# ---------------------------------------------------------
# FEATURE EXTRACTION
# ---------------------------------------------------------
def extract_features(file_path):

    y, sr = librosa.load(
        file_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        fmax=1000
    )

    mel_db = librosa.power_to_db(
        mel,
        ref=np.max
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=sr,
        fmin=50,
        n_bands=6
    )

    features = np.concatenate([
        mel_db.flatten(),
        contrast.flatten()
    ])

    return features


# ---------------------------------------------------------
# LABEL MAPPING
# ---------------------------------------------------------
def map_label_from_path(path):

    p = path.lower()

    if "background" in p:
        return "background"

    if "pre" in p and "feed" in p:
        return "pre-feeding"

    if "post" in p and "feed" in p:
        return "post-feeding"

    if "feeding" in p:
        return "feeding"

    return None


# ---------------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------------
X = []
y = []

print("Loading dataset...")

for root, dirs, files in os.walk(DATASET_PATH):

    for file in files:

        if file.endswith(".wav"):

            file_path = os.path.join(root, file)

            label = map_label_from_path(root)

            if label is None:
                continue

            try:
                features = extract_features(file_path)

                X.append(features)
                y.append(label)

            except Exception as e:
                print(f"Error processing {file_path}")
                print(e)

X = np.array(X)
y = np.array(y)

print("\nLoaded samples:", X.shape[0])
print("Feature dimension:", X.shape[1])
print("Classes found:", sorted(set(y)))


# ---------------------------------------------------------
# ENCODING + SCALING
# ---------------------------------------------------------
le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ---------------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)


# ---------------------------------------------------------
# TRAIN SVM
# ---------------------------------------------------------
print("\nTraining SVM...")

svm = SVC(
    kernel="rbf",
    probability=True,
    random_state=42
)

start_train = time.time()

svm.fit(
    X_train,
    y_train
)

train_time = time.time() - start_train


# ---------------------------------------------------------
# INFERENCE
# ---------------------------------------------------------
start_inf = time.time()

y_pred = svm.predict(X_test)

inf_time = (
    time.time() - start_inf
) / len(X_test)

y_prob = svm.predict_proba(X_test)


# ---------------------------------------------------------
# METRICS
# ---------------------------------------------------------
acc = accuracy_score(y_test, y_pred)

prec = precision_score(
    y_test,
    y_pred,
    average="macro"
)

rec = recall_score(
    y_test,
    y_pred,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

try:
    multiclass_auc = roc_auc_score(
        y_test,
        y_prob,
        multi_class="ovr"
    )
except Exception:
    multiclass_auc = None

cm = confusion_matrix(
    y_test,
    y_pred
)


# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------
print("\n-----------------------------")
print("SVM RESULTS")
print("-----------------------------")
print("Training time:", round(train_time, 4), "s")
print("Inference time/sample:", round(inf_time * 1000, 4), "ms")
print("Accuracy:", round(acc, 4))
print("Precision:", round(prec, 4))
print("Recall:", round(rec, 4))
print("F1-score:", round(f1, 4))
print("Multiclass ROC-AUC:", multiclass_auc)

print("\nConfusion Matrix:")
print(cm)


# ---------------------------------------------------------
# OUTPUT DIRECTORY
# ---------------------------------------------------------
os.makedirs(
    "seabream_pictures",
    exist_ok=True
)


# ---------------------------------------------------------
# CONFUSION MATRIX
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="viridis",
    xticklabels=le.classes_,
    yticklabels=le.classes_,
    linewidths=0.5,
    linecolor="gray"
)

plt.title("Confusion Matrix - SVM")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()

plt.savefig(
    "seabream_pictures/confusion_matrix_svm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# ROC OVR
# ---------------------------------------------------------
n_classes = len(le.classes_)

y_test_bin = label_binarize(
    y_test,
    classes=np.arange(n_classes)
)

fpr = {}
tpr = {}
auc_scores = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    auc_scores[i] = auc(
        fpr[i],
        tpr[i]
    )


# ---------------------------------------------------------
# ROC FIGURE
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))

colors = [
    "blue",
    "orange",
    "green",
    "red"
]

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i % len(colors)],
        lw=2,
        label=f"{le.classes_[i]} (AUC = {auc_scores.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    "k--",
    lw=1.5,
    label="Random classifier"
)

plt.xlim([0, 1])
plt.ylim([0, 1.05])

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("One-vs-Rest ROC Curves - SVM")

plt.legend(loc="lower right")

plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    "seabream_pictures/roc_ovr.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ---------------------------------------------------------
# PRINT AUC VALUES
# ---------------------------------------------------------
print("\nPer-class AUC")

for i in range(n_classes):
    print(
        f"{le.classes_[i]} : {auc_scores.4f}"
    )

print("\nSaved files:")
print("seabream_pictures/confusion_matrix_svm.png")
print("seabream_pictures/roc_ovr.png")

SyntaxError: invalid decimal literal (1279515207.py, line 335)